In [5]:
import os
import pandas as pd

CORE_TSV = "/ssd/wangab/DeepDeconv/Result/train_set/core_train_dedup.tsv"
PRSM_DIR = "/ssd/wangab/DeepDeconv/Result/prsm"   # 这里放着 prsm0.csv, prsm1.csv, ...
PRSM_PATTERN = "prsm{pid}.csv"

# 1) 读 core_train_dedup.tsv
core = pd.read_csv(CORE_TSV, sep="\t", low_memory=False)

if "Prsm ID" not in core.columns:
    raise KeyError(f"core_train_dedup.tsv 里找不到 'Prsm ID' 列，实际列为: {core.columns.tolist()}")

# 规范化为 int（你的示例里是整数）
core["Prsm ID"] = pd.to_numeric(core["Prsm ID"], errors="coerce").astype("Int64")

pids = core["Prsm ID"].dropna().astype(int).unique().tolist()
pids.sort()

# 2) 逐个读取 prsm{ID}.csv 并拼接
prsm_pieces = []
missing = []

for pid in pids:
    csv_path = os.path.join(PRSM_DIR, PRSM_PATTERN.format(pid=pid))
    if not os.path.exists(csv_path):
        missing.append(csv_path)
        continue

    df_pr = pd.read_csv(csv_path, low_memory=False)

    # 安全检查：确保 prsm_id 列存在
    if "prsm_id" not in df_pr.columns:
        # 没有就补一个（按文件名推断）
        df_pr["prsm_id"] = pid

    prsm_pieces.append(df_pr)

prsm_all = pd.concat(prsm_pieces, ignore_index=True) if prsm_pieces else pd.DataFrame()

print(f"core 行数: {len(core)}")
print(f"唯一 Prsm ID 数: {len(pids)}")
print(f"成功读取 prsmX.csv 数: {len(prsm_pieces)}")
print(f"缺失 prsmX.csv 数: {len(missing)}")
if missing:
    print("缺失示例(前5个):")
    for p in missing[:5]:
        print("  ", p)

print(len(prsm_all))  # 查看 prsm_all 的前几行，确认数据结构
# # 3) 可选：把 prsm_all 合并回 core（一对多：core一行可能对应prsm_all多行碎片）
# # 如果只想要 prsm_all，这一步可以不要
# core_with_prsm = core.merge(
#     prsm_all,
#     left_on="Prsm ID",
#     right_on="prsm_id",
#     how="left",
#     suffixes=("", "__prsmcsv"),
# )

# # 4) 保存结果（可选）
# prsm_all.to_csv("/ssd/wangab/DeepDeconv/Result/train_set/prsm_all_from_core.csv", index=False)
# core_with_prsm.to_csv("/ssd/wangab/DeepDeconv/Result/train_set/core_train_dedup_with_prsmcsv.tsv", sep="\t", index=False)

core 行数: 676
唯一 Prsm ID 数: 676
成功读取 prsmX.csv 数: 676
缺失 prsmX.csv 数: 0
21444


In [3]:
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# -----------------------
# 0) 配置
# -----------------------
@dataclass
class TrainConfig:
    core_tsv: str = "/ssd/wangab/DeepDeconv/Result/train_set/core_train_dedup.tsv"
    prsm_dir: str = "/ssd/wangab/DeepDeconv/Result/prsm"
    prsm_pattern: str = "prsm{pid}.csv"

    # 输入
    max_len: int = 350  # proteoform 最大长度（氨基酸数）；超过截断
    aa_embed_dim: int = 64
    cond_embed_dim: int = 16  # 实验条件 embedding（碎裂方式、能量离散桶）

    # encoder
    lstm_hidden: int = 256
    lstm_layers: int = 2
    dropout: float = 0.2

    # heads
    head_hidden: int = 128
    charge_num_classes: int = 10  # peak_charge 1..9 => 9类(含0 padding)；这里先给 10

    # 训练
    batch_size: int = 32
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 10
    num_workers: int = 4
    seed: int = 1337
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # loss 权重（按需调整）
    w_intensity: float = 1.0
    w_dmass: float = 0.2
    w_charge: float = 0.2

    # 强度是否做归一化
    intensity_log1p: bool = True
    intensity_per_prsm_norm: bool = True


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# -----------------------
# 1) 字典与特征
# -----------------------
AA_VOCAB = list("ACDEFGHIKLMNPQRSTVWY")  # 20 标准氨基酸
AA_TO_ID = {aa: i + 1 for i, aa in enumerate(AA_VOCAB)}  # 0 留给 padding/unknown
PAD_ID = 0

FRAG_TO_ID = {
    "CID": 1,
    "HCD": 2,
    "ETD": 3,
    "ECD": 4,
    "EThcD": 5,
    "UVPD": 6,
    "UNKNOWN": 0,
}

def encode_sequence(seq: str, max_len: int) -> Tuple[np.ndarray, np.ndarray]:
    """返回 token_ids (max_len,), mask (max_len,)"""
    seq = str(seq)
    ids = np.zeros((max_len,), dtype=np.int64)
    msk = np.zeros((max_len,), dtype=np.float32)
    for i, ch in enumerate(seq[:max_len]):
        ids[i] = AA_TO_ID.get(ch, 0)
        msk[i] = 1.0
    return ids, msk

def encode_fragmentation(frag: str) -> int:
    if pd.isna(frag):
        return FRAG_TO_ID["UNKNOWN"]
    frag = str(frag).strip()
    return FRAG_TO_ID.get(frag, FRAG_TO_ID["UNKNOWN"])

def encode_energy_to_bucket(energy: Optional[float]) -> int:
    """
    碎裂能量离散桶。你当前 core_train_dedup.tsv 看起来没有能量列，
    所以这里允许 None -> 0。
    """
    if energy is None or pd.isna(energy):
        return 0
    e = float(energy)
    # 简单分桶示例：0..100 -> 1..10
    b = int(min(9, max(0, math.floor(e / 10.0))))
    return b + 1


# -----------------------
# 2) 从 prsmX.csv 构建 fragment-level 标签序列
# -----------------------
def build_targets_from_prsm_csv(
    df_pr: pd.DataFrame,
    seq_len: int,
    max_len: int,
    cfg: TrainConfig,
) -> Dict[str, np.ndarray]:
    """
    构造 shape=(max_len,) 或 (max_len-1,) 的 labels。
    这里用 ion_position -> token position（1-based）映射到序列位置:
      position_index = ion_position-1
    标签：
      intensity: float32, padding=0
      dmass: float32, padding=0
      charge: int64, padding=0 (ignore_index)
      frag_mask: float32,表示该位置是否有监督 fragment
    """
    L = min(seq_len, max_len)
    # fragment通常对应切位，最大 L-1（但为简单，这里用 L）
    T = L

    intensity = np.zeros((T,), dtype=np.float32)
    dmass = np.zeros((T,), dtype=np.float32)
    charge = np.zeros((T,), dtype=np.int64)
    frag_mask = np.zeros((T,), dtype=np.float32)

    if df_pr is None or len(df_pr) == 0:
        return dict(intensity=intensity, dmass=dmass, charge=charge, frag_mask=frag_mask)

    # 取需要的列（你的 prsm0.csv 有 peak_intensity, peak_charge, mass_error, ppm, ion_position）
    # delta_mass 先用 mass_error（绝对质量误差）或 ppm
    use_dmass_col = "mass_error" if "mass_error" in df_pr.columns else ("ppm" if "ppm" in df_pr.columns else None)

    # 对每个 ion_position 聚合（同一位置可能多个峰/多条匹配）
    # 这里 intensity 取 max；dmass 取 mean；charge 取众数
    grouped = df_pr.dropna(subset=["ion_position"]).copy()
    grouped["ion_position"] = pd.to_numeric(grouped["ion_position"], errors="coerce").astype("Int64")
    grouped = grouped.dropna(subset=["ion_position"])

    for pos, g in grouped.groupby("ion_position"):
        pos = int(pos)
        idx = pos - 1
        if idx < 0 or idx >= T:
            continue

        if "peak_intensity" in g.columns:
            vals = pd.to_numeric(g["peak_intensity"], errors="coerce").dropna()
            if len(vals) > 0:
                intensity[idx] = float(vals.max())

        if use_dmass_col is not None:
            vals = pd.to_numeric(g[use_dmass_col], errors="coerce").dropna()
            if len(vals) > 0:
                dmass[idx] = float(vals.mean())

        if "peak_charge" in g.columns:
            vals = pd.to_numeric(g["peak_charge"], errors="coerce").dropna()
            if len(vals) > 0:
                # 众数
                ch = int(vals.round().astype(int).mode().iloc[0])
                ch = max(0, min(cfg.charge_num_classes - 1, ch))
                charge[idx] = ch

        frag_mask[idx] = 1.0

    # 强度预处理（可选）
    if cfg.intensity_log1p:
        intensity = np.log1p(intensity)

    if cfg.intensity_per_prsm_norm:
        m = intensity.max()
        if m > 0:
            intensity = intensity / m

    return dict(intensity=intensity, dmass=dmass, charge=charge, frag_mask=frag_mask)


# -----------------------
# 3) Dataset
# -----------------------
class CorePrsmDataset(Dataset):
    def __init__(self, core_df: pd.DataFrame, cfg: TrainConfig, energy_col: Optional[str] = None):
        self.core = core_df.reset_index(drop=True)
        self.cfg = cfg
        self.energy_col = energy_col

        required = ["Prsm ID", "Proteoform", "Fragmentation"]
        for c in required:
            if c not in self.core.columns:
                raise KeyError(f"core_train_dedup 缺少列: {c}")

        # 预处理 Prsm ID
        self.core["Prsm ID"] = pd.to_numeric(self.core["Prsm ID"], errors="coerce").astype("Int64")
        self.core = self.core.dropna(subset=["Prsm ID"]).reset_index(drop=True)

    def __len__(self):
        return len(self.core)

    def _load_prsm_csv(self, pid: int) -> pd.DataFrame:
        path = os.path.join(self.cfg.prsm_dir, self.cfg.prsm_pattern.format(pid=pid))
        if not os.path.exists(path):
            return pd.DataFrame()
        return pd.read_csv(path, low_memory=False)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.core.iloc[idx]
        pid = int(row["Prsm ID"])
        seq = str(row["Proteoform"])
        frag = row["Fragmentation"]

        # 能量（如果你有 NCE/CE 列，把 energy_col 传进来）
        energy = None
        if self.energy_col is not None and self.energy_col in self.core.columns:
            energy = row[self.energy_col]

        seq_ids, seq_mask = encode_sequence(seq, self.cfg.max_len)
        frag_id = encode_fragmentation(frag)
        e_bucket = encode_energy_to_bucket(energy)

        df_pr = self._load_prsm_csv(pid)
        targets = build_targets_from_prsm_csv(df_pr, seq_len=len(seq), max_len=self.cfg.max_len, cfg=self.cfg)

        return {
            "seq_ids": torch.from_numpy(seq_ids),                 # (T,)
            "seq_mask": torch.from_numpy(seq_mask),               # (T,)
            "frag_id": torch.tensor(frag_id, dtype=torch.long),   # ()
            "e_bucket": torch.tensor(e_bucket, dtype=torch.long), # ()

            "y_intensity": torch.from_numpy(targets["intensity"]),# (T,)
            "y_dmass": torch.from_numpy(targets["dmass"]),        # (T,)
            "y_charge": torch.from_numpy(targets["charge"]),      # (T,)
            "frag_mask": torch.from_numpy(targets["frag_mask"]),  # (T,)
        }


def collate_batch(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    # 全部都已 pad 到 max_len，所以直接 stack
    out = {}
    for k in batch[0].keys():
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out


# -----------------------
# 4) 模型：BiLSTM encoder + 多头
# -----------------------
class BiLSTMFragmentPredictor(nn.Module):
    def __init__(self, cfg: TrainConfig, aa_vocab_size: int = 21):
        super().__init__()
        self.cfg = cfg

        self.aa_emb = nn.Embedding(aa_vocab_size, cfg.aa_embed_dim, padding_idx=PAD_ID)

        # fragmentation embedding + energy bucket embedding
        self.frag_emb = nn.Embedding(max(FRAG_TO_ID.values()) + 1, cfg.cond_embed_dim)
        self.energy_emb = nn.Embedding(11, cfg.cond_embed_dim)  # 0..10 bucket

        in_dim = cfg.aa_embed_dim + 2 * cfg.cond_embed_dim

        self.encoder = nn.LSTM(
            input_size=in_dim,
            hidden_size=cfg.lstm_hidden,
            num_layers=cfg.lstm_layers,
            dropout=cfg.dropout if cfg.lstm_layers > 1 else 0.0,
            bidirectional=True,
            batch_first=True,
        )

        enc_dim = 2 * cfg.lstm_hidden

        # 多任务头：按时间步输出
        self.intensity_head = nn.Sequential(
            nn.Linear(enc_dim, cfg.head_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden, 1),
        )
        self.dmass_head = nn.Sequential(
            nn.Linear(enc_dim, cfg.head_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden, 1),
        )
        self.charge_head = nn.Sequential(
            nn.Linear(enc_dim, cfg.head_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden, cfg.charge_num_classes),
        )

    def forward(self, seq_ids, seq_mask, frag_id, e_bucket):
        """
        seq_ids: (B,T)
        seq_mask: (B,T) 1=valid aa, 0=pad
        frag_id: (B,)
        e_bucket: (B,)
        """
        B, T = seq_ids.shape

        aa = self.aa_emb(seq_ids)  # (B,T,E)

        frag = self.frag_emb(frag_id).unsqueeze(1).expand(B, T, -1)
        ene = self.energy_emb(e_bucket).unsqueeze(1).expand(B, T, -1)

        x = torch.cat([aa, frag, ene], dim=-1)  # (B,T,in_dim)

        # pack for LSTM
        lengths = seq_mask.sum(dim=1).long().clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.encoder(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=T)  # (B,T,2H)

        pred_int = self.intensity_head(h).squeeze(-1)  # (B,T)
        pred_dm = self.dmass_head(h).squeeze(-1)       # (B,T)
        pred_ch = self.charge_head(h)                  # (B,T,C)

        return pred_int, pred_dm, pred_ch


# -----------------------
# 5) Loss（mask）
# -----------------------
def masked_l1(pred, target, mask):
    # mask: (B,T) float
    diff = (pred - target).abs() * mask
    denom = mask.sum().clamp(min=1.0)
    return diff.sum() / denom

def masked_ce(logits, target, mask, ignore_index: int = 0):
    """
    target: (B,T) long，padding=0
    mask: (B,T) float，1=use this position
    """
    B, T, C = logits.shape
    logits = logits.reshape(B * T, C)
    target = target.reshape(B * T)
    mask = mask.reshape(B * T)

    # 先算每个 token 的 CE，再按 mask 求均值
    ce = F.cross_entropy(logits, target, reduction="none", ignore_index=ignore_index)  # (B*T,)
    ce = ce * mask
    denom = mask.sum().clamp(min=1.0)
    return ce.sum() / denom


# -----------------------
# 6) 训练循环
# -----------------------
def train_one_epoch(model, loader, optim, cfg: TrainConfig):
    model.train()
    total = 0.0
    nstep = 0
    for batch in loader:
        for k in batch:
            batch[k] = batch[k].to(cfg.device)

        pred_int, pred_dm, pred_ch = model(
            batch["seq_ids"], batch["seq_mask"], batch["frag_id"], batch["e_bucket"]
        )

        # 只在 seq_mask & frag_mask 有监督的位置算 loss
        mask = batch["seq_mask"] * batch["frag_mask"]

        loss_int = masked_l1(pred_int, batch["y_intensity"], mask)
        loss_dm = masked_l1(pred_dm, batch["y_dmass"], mask)
        loss_ch = masked_ce(pred_ch, batch["y_charge"], mask, ignore_index=0)

        loss = cfg.w_intensity * loss_int + cfg.w_dmass * loss_dm + cfg.w_charge * loss_ch

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()

        total += float(loss.item())
        nstep += 1

    return total / max(1, nstep)


@torch.no_grad()
def validate(model, loader, cfg: TrainConfig):
    model.eval()
    total = 0.0
    nstep = 0
    for batch in loader:
        for k in batch:
            batch[k] = batch[k].to(cfg.device)

        pred_int, pred_dm, pred_ch = model(
            batch["seq_ids"], batch["seq_mask"], batch["frag_id"], batch["e_bucket"]
        )
        mask = batch["seq_mask"] * batch["frag_mask"]

        loss_int = masked_l1(pred_int, batch["y_intensity"], mask)
        loss_dm = masked_l1(pred_dm, batch["y_dmass"], mask)
        loss_ch = masked_ce(pred_ch, batch["y_charge"], mask, ignore_index=0)
        loss = cfg.w_intensity * loss_int + cfg.w_dmass * loss_dm + cfg.w_charge * loss_ch

        total += float(loss.item())
        nstep += 1
    return total / max(1, nstep)


# def main():
cfg = TrainConfig()
seed_everything(cfg.seed)

core = pd.read_csv(cfg.core_tsv, sep="\t", low_memory=False)

# 简单划分 train/val（按行随机；更严谨可按 Protein accession 或文件名分层）
idx = np.arange(len(core))
np.random.shuffle(idx)
split = int(0.9 * len(idx))
train_df = core.iloc[idx[:split]].reset_index(drop=True)
val_df = core.iloc[idx[split:]].reset_index(drop=True)

train_ds = CorePrsmDataset(train_df, cfg, energy_col=None)  # 如有能量列，填 energy_col="NCE" 等
val_ds = CorePrsmDataset(val_df, cfg, energy_col=None)
print(train_ds[0])  # 看看第一个样本的结构和数据
# exit()
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device.startswith("cuda")),
    collate_fn=collate_batch,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device.startswith("cuda")),
    collate_fn=collate_batch,
)

# model = BiLSTMFragmentPredictor(cfg).to(cfg.device)
# optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# best = float("inf")
# for epoch in range(1, cfg.epochs + 1):
#     tr = train_one_epoch(model, train_loader, optim, cfg)
#     va = validate(model, val_loader, cfg)
#     print(f"epoch {epoch:03d} train_loss={tr:.5f} val_loss={va:.5f}")

#     if va < best:
#         best = va
#         os.makedirs("./checkpoints", exist_ok=True)
#         torch.save(
#             {"model": model.state_dict(), "cfg": cfg.__dict__},
#             "./checkpoints/bilstm_frag_multitask.pt",
#         )

# print("best_val_loss:", best)


# if __name__ == "__main__":
#     main()

{'seq_ids': tensor([ 8,  0, 11, 12, 16,  5, 18, 12,  3,  8,  5,  4, 15,  8,  1,  6,  4,  1,
        16, 15, 10,  1,  7, 20, 12,  9, 15, 16, 17,  8,  0, 17,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0, 